# AI Company Simulator

This notebook builds and runs a simple multi-agent AI company simulation in Google Colab.

What this notebook does:
- Creates the full project structure from scratch
- Installs the only dependency: `requests`
- Lets you add your OpenRouter API key
- Runs a sample task through the manager and worker agents
- Optionally launches the CLI app


## Step 1: Create the project files

Run the next cell once. It writes every required file into `/content/ai_company`.

In [ ]:
from pathlib import Path
import textwrap

base = Path('/content/ai_company')
files = {
    'main.py': '''
"""CLI entrypoint for the AI Company Simulator."""

try:
    from ai_company.config import APP_NAME, EXIT_COMMANDS, LOGGER
    from ai_company.core.memory import MemoryStore
    from ai_company.core.router import TaskRouter
except ImportError:
    from config import APP_NAME, EXIT_COMMANDS, LOGGER
    from core.memory import MemoryStore
    from core.router import TaskRouter


def print_banner() -> None:
    """Show a short welcome message for CLI users."""
    print('=' * 60)
    print(f'{APP_NAME}')
    print('Type a business task and the simulator will route it.')
    print(f"Exit anytime with: {', '.join(sorted(EXIT_COMMANDS))}")
    print('=' * 60)


def main() -> None:
    """Run a simple REPL loop for the simulator."""
    memory = MemoryStore()
    router = TaskRouter()

    print_banner()

    while True:
        user_input = input('\\nTask> ').strip()

        if not user_input:
            print('Please enter a task for the company.')
            continue

        if user_input.lower() in EXIT_COMMANDS:
            print('\\nSession ended. Thanks for using the simulator.')
            break

        LOGGER.info('Received new task from user')
        result = router.handle_task(user_input)
        memory.add_entry(task=user_input, response=result['response'])

        print('\\nDepartment:', result['department'].title())
        print('Response:')
        print(result['response'])


if __name__ == '__main__':
    main()
''',
    'config.py': '''
"""Configuration values used across the project."""

import logging
import os


APP_NAME = 'AI Company Simulator'
OPENROUTER_API_URL = 'https://openrouter.ai/api/v1/chat/completions'
OPENROUTER_MODEL = 'openrouter/free'
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

REQUEST_TIMEOUT = 45
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 2

EXIT_COMMANDS = {'exit', 'quit', 'q'}


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
)
LOGGER = logging.getLogger(APP_NAME)
''',
    'llm.py': '''
"""Lightweight OpenRouter client built with requests."""

import os
import time
from typing import Any, Dict

import requests

try:
    from ai_company.config import (
        LOGGER,
        MAX_RETRIES,
        OPENROUTER_API_URL,
        OPENROUTER_MODEL,
        REQUEST_TIMEOUT,
        RETRY_DELAY_SECONDS,
    )
except ImportError:
    from config import (
        LOGGER,
        MAX_RETRIES,
        OPENROUTER_API_URL,
        OPENROUTER_MODEL,
        REQUEST_TIMEOUT,
        RETRY_DELAY_SECONDS,
    )


def _extract_text(data: Dict[str, Any]) -> str:
    """Safely pull assistant text from an OpenRouter response."""
    try:
        content = data['choices'][0]['message']['content']
    except (KeyError, IndexError, TypeError):
        return 'I could not parse the model response.'

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and item.get('type') == 'text':
                parts.append(item.get('text', ''))
        return '\\n'.join(part for part in parts if part).strip() or 'No text returned.'

    return str(content).strip() or 'No text returned.'


def call_llm(prompt: str) -> str:
    """Send a prompt to OpenRouter and return only the text output."""
    api_key = os.getenv('OPENROUTER_API_KEY', '')

    if not api_key:
        return (
            'OpenRouter API key not found. Set the OPENROUTER_API_KEY '
            'environment variable and try again.'
        )

    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
        'HTTP-Referer': 'https://colab.research.google.com/',
        'X-Title': 'AI Company Simulator',
    }
    payload = {
        'model': OPENROUTER_MODEL,
        'messages': [
            {
                'role': 'user',
                'content': prompt,
            }
        ],
        'temperature': 0.3,
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            LOGGER.info('Calling OpenRouter (attempt %s/%s)', attempt, MAX_RETRIES)
            response = requests.post(
                OPENROUTER_API_URL,
                headers=headers,
                json=payload,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            data = response.json()
            return _extract_text(data)
        except requests.exceptions.RequestException as exc:
            LOGGER.warning('OpenRouter request failed: %s', exc)
            if attempt == MAX_RETRIES:
                return f'LLM request failed after {MAX_RETRIES} attempts: {exc}'
            time.sleep(RETRY_DELAY_SECONDS)
        except ValueError as exc:
            LOGGER.warning('Invalid JSON from OpenRouter: %s', exc)
            return f'LLM returned an invalid JSON response: {exc}'

    return 'Unexpected error while calling the language model.'
''',
    'agents/__init__.py': '''"""Agent modules for the AI Company Simulator."""''',
    'agents/manager.py': '''
"""Manager agent that classifies tasks into departments."""

try:
    from ai_company.llm import call_llm
    from ai_company.utils.prompts import manager_prompt
except ImportError:
    from llm import call_llm
    from utils.prompts import manager_prompt


class ManagerAgent:
    """Decides which worker agent should handle a task."""

    valid_departments = {'marketing', 'support', 'analytics'}

    def decide(self, task: str) -> str:
        """Return the best department for the given task."""
        prompt = manager_prompt(task)
        raw_decision = call_llm(prompt).strip().lower()

        for department in self.valid_departments:
            if department in raw_decision:
                return department

        return 'support'
''',
    'agents/marketing.py': '''
"""Marketing worker agent."""

try:
    from ai_company.llm import call_llm
    from ai_company.utils.prompts import marketing_prompt
except ImportError:
    from llm import call_llm
    from utils.prompts import marketing_prompt


class MarketingAgent:
    """Handles marketing and growth-related tasks."""

    name = 'marketing'

    def process(self, task: str) -> str:
        """Generate a marketing-focused response."""
        return call_llm(marketing_prompt(task))
''',
    'agents/support.py': '''
"""Support worker agent."""

try:
    from ai_company.llm import call_llm
    from ai_company.utils.prompts import support_prompt
except ImportError:
    from llm import call_llm
    from utils.prompts import support_prompt


class SupportAgent:
    """Handles customer support and response tasks."""

    name = 'support'

    def process(self, task: str) -> str:
        """Generate a support-focused response."""
        return call_llm(support_prompt(task))
''',
    'agents/analytics.py': '''
"""Analytics worker agent."""

try:
    from ai_company.llm import call_llm
    from ai_company.utils.prompts import analytics_prompt
except ImportError:
    from llm import call_llm
    from utils.prompts import analytics_prompt


class AnalyticsAgent:
    """Handles data, reporting, and analysis tasks."""

    name = 'analytics'

    def process(self, task: str) -> str:
        """Generate an analytics-focused response."""
        return call_llm(analytics_prompt(task))
''',
    'core/__init__.py': '''"""Core modules for routing and memory."""''',
    'core/router.py': '''
"""Task routing logic for the simulator."""

try:
    from ai_company.agents.analytics import AnalyticsAgent
    from ai_company.agents.manager import ManagerAgent
    from ai_company.agents.marketing import MarketingAgent
    from ai_company.agents.support import SupportAgent
    from ai_company.config import LOGGER
except ImportError:
    from agents.analytics import AnalyticsAgent
    from agents.manager import ManagerAgent
    from agents.marketing import MarketingAgent
    from agents.support import SupportAgent
    from config import LOGGER


class TaskRouter:
    """Routes tasks from the manager agent to worker agents."""

    def __init__(self) -> None:
        self.manager = ManagerAgent()
        self.workers = {
            'marketing': MarketingAgent(),
            'support': SupportAgent(),
            'analytics': AnalyticsAgent(),
        }

    def handle_task(self, task: str) -> dict:
        """Send a task to the correct department and return the result."""
        department = self.manager.decide(task)
        worker = self.workers.get(department, self.workers['support'])

        LOGGER.info('Task routed to %s', department)
        response = worker.process(task)
        return {
            'department': department,
            'response': response,
        }
''',
    'core/memory.py': '''
"""Simple in-memory storage for tasks and responses."""

from datetime import datetime
from typing import Dict, List


class MemoryStore:
    """Stores simulator history in a Python list."""

    def __init__(self) -> None:
        self.history: List[Dict[str, str]] = []

    def add_entry(self, task: str, response: str) -> None:
        """Save a task, response, and timestamp."""
        self.history.append(
            {
                'task': task,
                'response': response,
                'timestamp': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            }
        )

    def get_history(self) -> List[Dict[str, str]]:
        """Return the full conversation history."""
        return self.history
''',
    'utils/__init__.py': '''"""Prompt helpers for the AI Company Simulator."""''',
    'utils/prompts.py': '''
"""Reusable prompt templates for each agent."""


def manager_prompt(task: str) -> str:
    """Prompt that forces structured department selection."""
    return f"""
You are the Manager Agent for an AI company simulation.

Your job is to classify the user's request into exactly one department.

Valid departments:
- marketing
- support
- analytics

Rules:
- Reply with only one word.
- Do not explain your answer.
- Choose the single best department.

User task:
{task}
""".strip()


def marketing_prompt(task: str) -> str:
    """Prompt template for the marketing worker."""
    return f"""
You are the Marketing Agent in an AI company.

Your responsibilities:
- campaign ideas
- social media messaging
- product positioning
- audience engagement

Respond with:
1. A short analysis
2. A practical recommendation
3. A concise final answer

Task:
{task}
""".strip()


def support_prompt(task: str) -> str:
    """Prompt template for the support worker."""
    return f"""
You are the Support Agent in an AI company.

Your responsibilities:
- answer customer questions
- solve user issues
- write helpful and polite replies
- suggest next steps when needed

Respond with:
1. Issue understanding
2. Resolution steps
3. A clear user-facing answer

Task:
{task}
""".strip()


def analytics_prompt(task: str) -> str:
    """Prompt template for the analytics worker."""
    return f"""
You are the Analytics Agent in an AI company.

Your responsibilities:
- interpret metrics
- summarize trends
- make data-driven recommendations
- highlight risks or opportunities

Respond with:
1. Key insight
2. Supporting reasoning
3. Actionable recommendation

Task:
{task}
""".strip()
''',
    'requirements.txt': '''requests\n''',
    'README.md': '''
# AI Company Simulator

A simple, modular multi-agent AI company simulation built in Python for Google Colab. The project uses OpenRouter with the `requests` library and routes each user task through a manager agent before sending it to a specialized worker agent.

## Features

- Manager agent for task classification
- Worker agents for marketing, support, and analytics
- Central router for delegation
- In-memory task history
- OpenRouter integration with retry logic
- CLI interaction for easy demos and workshops

## Project Structure

```text
ai_company/
├── main.py
├── config.py
├── llm.py
├── agents/
│   ├── manager.py
│   ├── marketing.py
│   ├── support.py
│   └── analytics.py
├── core/
│   ├── router.py
│   └── memory.py
├── utils/
│   └── prompts.py
├── requirements.txt
└── README.md
```

## How It Works

1. The user enters a task in the CLI.
2. The manager agent classifies the task as `marketing`, `support`, or `analytics`.
3. The router sends the task to the matching worker agent.
4. The worker agent calls OpenRouter and generates a response.
5. The task, response, and timestamp are saved to memory.

## Architecture Diagram

```text
User Input
   |
   v
main.py
   |
   v
TaskRouter
   |
   v
ManagerAgent
   |
   +--> MarketingAgent
   |
   +--> SupportAgent
   |
   +--> AnalyticsAgent
   |
   v
MemoryStore
   |
   v
CLI Output
```
''',
}

for relative_path, content in files.items():
    file_path = base / relative_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(textwrap.dedent(content).lstrip('\n'), encoding='utf-8')

(base / '__init__.py').write_text('"""AI Company Simulator package."""\n', encoding='utf-8')

print('Project created at:', base)
for path in sorted(base.rglob('*')):
    if path.is_file():
        print(path)


## Step 2: Install requirements

In [ ]:
%cd /content/ai_company
!pip install -r requirements.txt

## Step 3: Add your OpenRouter API key

Use the hidden input below so your key does not stay visible in the notebook.

In [ ]:
from getpass import getpass
import os

os.environ['OPENROUTER_API_KEY'] = getpass('Paste your OpenRouter API key: ').strip()
print('API key loaded:', bool(os.environ.get('OPENROUTER_API_KEY')))

## Step 4: Test the API connection

Run this once to confirm your key works and OpenRouter can return a response.

In [ ]:
import os
import requests

api_key = os.environ['OPENROUTER_API_KEY']

url = 'https://openrouter.ai/api/v1/chat/completions'
headers = {
    'Authorization': f'Bearer {api_key}',
    'Content-Type': 'application/json',
    'HTTP-Referer': 'https://colab.research.google.com/',
    'X-Title': 'AI Company Simulator',
}
payload = {
    'model': 'openrouter/free',
    'messages': [
        {'role': 'user', 'content': 'Say hello in one short sentence.'}
    ],
    'temperature': 0.3,
}

response = requests.post(url, headers=headers, json=payload, timeout=45)
print('Status code:', response.status_code)
print(response.text[:500])


## Step 5: Run one sample task

If Step 4 returned `Status code: 200`, run this cell to test the multi-agent workflow.

In [ ]:
import sys

for name in list(sys.modules):
    if (
        name == 'config'
        or name == 'llm'
        or name.startswith('agents')
        or name.startswith('core')
        or name.startswith('utils')
        or name.startswith('ai_company')
    ):
        sys.modules.pop(name, None)

from core.router import TaskRouter
from core.memory import MemoryStore

router = TaskRouter()
memory = MemoryStore()

task = 'Create a short launch campaign idea for our AI note-taking app.'
result = router.handle_task(task)
memory.add_entry(task=task, response=result['response'])

print('Department:', result['department'])
print('Response:')
print(result['response'])
print('\nMemory:')
print(memory.get_history())


## Step 6: Try your own tasks

Edit the `task` string and rerun the cell.

In [ ]:
task = 'Analyze why website conversions dropped this week and suggest what to check first.'
result = router.handle_task(task)
memory.add_entry(task=task, response=result['response'])

print('Department:', result['department'])
print('Response:')
print(result['response'])


## Optional: Run the CLI app

Colab sometimes handles interactive `input()` less smoothly than a normal terminal, but you can still try it after the cells above work.

In [ ]:
%run main.py